# 04 — Split the proposed method into two explicit steps

This notebook shows the proposed method without hiding the pieces inside one wrapper.

The paper's proposed method is

\[
\texttt{weak\_pareto} = \text{weak fractional candidate library} + \text{best-subset Pareto-DE}.
\]

In the scripts this is available as one function, `run_weak_pareto_discovery(...)`.
Here we split it into the two conceptual stages:

1. **Weak library construction**: build the weak target and weak RHS feature columns.
2. **Best-subset Pareto-DE**: search over support size `c`, temporal order `alpha`, spatial orders `beta`, and coefficients.

The notebook also shows how to track the best equation found at every support size.


## 0. One editable setup cell

Change only the variables below to reproduce a single script-equivalent case.  If you use the same values here and in `scripts/run_all_methods.py`, the code path is the same.


In [ ]:
# Dataset/noise/profile controls.
dataset_name = "synthetic_time_space_fractional_RD"
noise_percent = 0.0
profile = "notebook"   # use "paper" for paper-quality dataset size and DE budget
seed = 0

# Candidate-library overrides. In notebook mode we use a small teaching library;
# in paper mode, leave them as None to use the canonical paper library.
cmax_override = 2 if profile == "notebook" else None
p_values_override = (0,) if profile == "notebook" else None

# Notebook mode is intentionally fast.  In paper mode, leave these as None so
# benchmark_spec(...) uses the dataset-specific paper defaults.
maxiter_override = 0 if profile == "notebook" else None
popsize_override = 2 if profile == "notebook" else None
weak_test_budget = "smoke" if profile == "notebook" else "paper"

# Output path is created after the project root is discovered.
output_subdir = f"results/notebook_split_weak_pareto/{dataset_name}/noise_{noise_percent:g}_seed_{seed}"

## 1. Load exactly the same benchmark spec used by the scripts

`benchmark_spec(...)` returns the data, canonical search configuration, and truth metadata for one dataset name.  The discovery method does **not** receive the truth; the truth is only used later to evaluate whether the recovered equation is correct.


In [ ]:
# Robust project-root discovery.
# This avoids failures when a Jupyter kernel is started in a directory that is
# later moved/deleted, in which case Path.cwd() itself can raise FileNotFoundError.
import os
import sys
from pathlib import Path


def find_fpde_project_root() -> Path:
    """Return the repository root containing dataset_configs.py and data/.

    Priority:
    1. FPDE_PROJECT_ROOT environment variable, if set.
    2. Current/PWD directories and their parents, if available.
    3. Common local search locations. This keeps notebooks runnable from
       project root, from notebooks/, and after opening a notebook from an IDE.
    """
    def looks_like_root(path: Path) -> bool:
        return (
            (path / "dataset_configs.py").is_file()
            and (path / "weak_pareto_fde_discovery.py").is_file()
            and (path / "data").is_dir()
        )

    candidates = []
    env_root = os.environ.get("FPDE_PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())

    # os.getcwd() can fail if the kernel's working directory was deleted.
    try:
        candidates.append(Path(os.getcwd()).expanduser())
    except FileNotFoundError:
        pass

    # PWD may still contain a useful absolute path even when os.getcwd() fails.
    pwd = os.environ.get("PWD")
    if pwd:
        candidates.append(Path(pwd).expanduser())

    # Also try the directory containing this notebook if Jupyter exposes it via env.
    for key in ("NOTEBOOK_DIR", "JUPYTER_SERVER_ROOT"):
        value = os.environ.get(key)
        if value:
            candidates.append(Path(value).expanduser())

    seen = set()
    for cand in candidates:
        try:
            cand = cand.resolve(strict=False)
        except Exception:
            continue
        for path in (cand, cand.parent, *cand.parents):
            if path in seen:
                continue
            seen.add(path)
            if looks_like_root(path):
                return path

    # Conservative bounded search over common project locations.
    search_roots = [
        Path.home() / "Desktop" / "research",
        Path.home() / "Desktop",
        Path.home(),
        Path("/mnt/data"),
    ]
    max_dirs = 5000
    for base in search_roots:
        if not base.exists():
            continue
        visited = 0
        for dirpath, dirnames, filenames in os.walk(base):
            visited += 1
            # Keep the search cheap and avoid hidden/cache directories.
            dirnames[:] = [d for d in dirnames if not d.startswith(".") and d not in {"__pycache__", ".ipynb_checkpoints"}]
            if "dataset_configs.py" in filenames and "weak_pareto_fde_discovery.py" in filenames:
                candidate = Path(dirpath)
                if looks_like_root(candidate):
                    return candidate
            if visited >= max_dirs:
                break

    raise FileNotFoundError(
        "Could not locate the fractional_pareto project root. "
        "Set FPDE_PROJECT_ROOT=/path/to/fractional_pareto_publication_ready_final "
        "or open the notebook from the project root/notebooks directory."
    )


ROOT = find_fpde_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")


import numpy as np
import pandas as pd

from dataset_configs import benchmark_spec


output_dir = ROOT / output_subdir
output_dir.mkdir(parents=True, exist_ok=True)
print("Output dir:", output_dir)

spec = benchmark_spec(
    dataset_name,
    data_dir=ROOT / "data",
    profile=profile,
    noise_percent=noise_percent,
    seed=seed,
    maxiter=maxiter_override,
    popsize=popsize_override,
    cmax=cmax_override,
    p_values=p_values_override,
)

data = spec["data"]
config = spec["config"]
truth_spec = spec["truth_spec"]
truth = truth_spec  # short alias used below for truth-only diagnostics
config.progress = False
config.progress_de = False

print(data.truth)
print("config:", {
    "profile": profile,
    "noise_percent": config.noise_percent,
    "seed": config.seed,
    "cmax": config.cmax,
    "p_values": config.p_values,
    "maxiter": config.maxiter,
    "popsize": config.popsize,
})


## 2. Step 1 — build the weak candidate library

This step creates the weak regression rows.  It does not select a model yet.

The weak library exposes the same optimizer-facing API as the vanilla library:

- `bank.target(alpha)` returns the weak LHS vector.
- `bank.library(p_tuple, beta_tuple)` returns weak RHS columns.

The difference is mathematical: the weak library applies fractional derivatives to test functions via adjoints, not to noisy data directly.


In [ ]:
from weak_pareto_fde_discovery import build_weak_candidate_library

bank = build_weak_candidate_library(
    data,
    config,
    test_budget=weak_test_budget,
    verbose=True,
)

print("Weak rows:", bank.n_points)
print("Time tests:", bank.time_tests.shape)
print("Space tests:", bank.space_tests.shape)
print("Time weak kind:", bank.time_kind)
print("Space weak kind:", bank.space_kind)
print("Time discretization:", bank.time_discretization)


## 3. Inspect a manually chosen weak feature matrix

Here we build the weak target and RHS matrix at the known benchmark structure. This is for learning and diagnostics only; the optimizer below will not be given the truth.


In [ ]:
alpha_probe = float(truth.expected_alpha)
p_tuple_probe = tuple(int(p) for p, _ in truth.expected_terms)
beta_tuple_probe = tuple(float(beta) for _, beta in truth.expected_terms)

y = bank.target(alpha_probe)
Theta = bank.library(p_tuple_probe, beta_tuple_probe)

print("Probe alpha:", alpha_probe)
print("Probe p_tuple:", p_tuple_probe)
print("Probe beta_tuple:", beta_tuple_probe)
print("Weak target shape:", y.shape)
print("Weak RHS matrix shape:", Theta.shape)
print("Target finite rows:", np.isfinite(y).sum(), "/", y.size)
print("RHS finite rows:", np.isfinite(Theta).all(axis=1).sum(), "/", Theta.shape[0])

# Simple coefficient fit at the probe structure, for diagnostics only.
coef, *_ = np.linalg.lstsq(Theta, y, rcond=None)
resid = y - Theta @ coef
print("Probe coefficients:", coef)
print("Probe relative residual:", np.linalg.norm(resid) / (np.linalg.norm(y) + 1e-14))

corr = np.corrcoef(Theta.T) if Theta.shape[1] > 1 else np.array([[1.0]])
pd.DataFrame(corr, index=[f"term {j}" for j in range(Theta.shape[1])], columns=[f"term {j}" for j in range(Theta.shape[1])])


## 4. Step 2 — create the best-subset Pareto-DE problem

This step wraps the already-built weak library in the same best-subset optimizer used by the script.  The support-size sweep then searches:

\[
c=1,2,\ldots,c_{\max}
\]

and records the best equation found for each support size.


In [ ]:
from weak_pareto_fde_discovery import build_best_subset_pareto_problem

problem = build_best_subset_pareto_problem(bank, config)
print("Train weak rows:", len(problem.train_idx))
print("Validation weak rows:", len(problem.val_idx))

# You can evaluate one candidate equation manually before running the sweep.
manual_model = problem.optimizer.evaluate(alpha_probe, p_tuple_probe, beta_tuple_probe)
print(manual_model.equation(digits=6))
print("manual val_rel_mse:", manual_model.val_rel_mse)


## 5. Run best-subset Pareto-DE and track the best equation at each support size

The output directory will contain:

- `all_models.csv`: every optimized fixed-support model;
- `best_by_c.csv`: best model at each support size;
- `support_size_progress.csv`: human-readable progress table for `c=1,2,...`;
- `summary.json`: full machine-readable summary;
- `selected_fde.json`: final selected equation.


In [ ]:
from weak_pareto_fde_discovery import run_best_subset_pareto_de

summary = run_best_subset_pareto_de(
    problem,
    config,
    data=data,
    output_dir=output_dir / "split_steps",
    verbose=True,
)

progress = pd.DataFrame(summary["support_size_progress"])
progress[["c", "equation", "val_rel_mse", "relative_improvement_from_previous_c"]]


## 6. Inspect the final selected equation

This is the model selected from the Pareto curve according to the canonical selection rule in the config.


In [ ]:
selected = summary["selected"]
print("Selected equation:")
print(selected["equation"])
print("Selected c:", selected["c"])
print("Selected alpha:", selected["alpha"])
print("Selected betas:", selected["beta_tuple"])
print("Validation relative MSE:", selected["val_rel_mse"])
print("Output files:")
for p in sorted((output_dir / "split_steps").glob("*")):
    print(" -", p.name)


## 7. Verify equivalence to the one-line proposed method wrapper

The one-line function `run_weak_pareto_discovery(...)` is what scripts use.  It should produce the same selected model because it simply calls the two steps above internally.


In [ ]:
from weak_pareto_fde_discovery import run_weak_pareto_discovery

wrapper_summary = run_weak_pareto_discovery(
    data,
    config,
    output_dir=output_dir / "one_line_wrapper",
    verbose=False,
    test_budget=weak_test_budget,
)

print("Split-step selected:", summary["selected"]["equation"])
print("Wrapper selected:   ", wrapper_summary["selected"]["equation"])
print("Same selected equation?", summary["selected"]["equation"] == wrapper_summary["selected"]["equation"])

pd.DataFrame(wrapper_summary["support_size_progress"])[["c", "equation", "val_rel_mse"]]


## 8. Takeaway

The proposed method is not a black box:

1. `build_weak_candidate_library(...)` constructs the weak fractional candidate library.
2. `build_best_subset_pareto_problem(...)` creates the optimizer object for that library.
3. `run_best_subset_pareto_de(...)` performs the best-subset Pareto-DE sweep.
4. `run_weak_pareto_discovery(...)` is just the script-friendly wrapper around these steps.

Use `support_size_progress.csv` to see what the optimizer found at each support size.
